# Track A — Bước A1a: Đo trần `Oracle XML ROI` trên `GC10-DET` (`duytung/03-gc10-dac-trung-vung-loi`)
- Dùng bounding box XML có sẵn của `GC10-DET` để crop vùng lỗi (`--roi_xml`, padding 15%), fine-tune với **`FT_EPOCHS=20`** (từ kết quả Bước A0), chạy `r0` (`s0..s4`) và đánh giá `P`, `N`, `NL`, `FTR1`, `Npre`.
- Tự động ghi đầy đủ **CPU, RAM, GPU, Peak VRAM, thời gian từng giai đoạn và tổng thời gian** vào `gc10_a1a_summary.txt` / `gc10_a1a_summary.json`.
- Cây quyết định: Nếu `bal >= 0.50` → khẳng định nền thép lấn át `[CLS]`, chuyển ngay sang **Bước A1b** (khoanh vùng lỗi không dùng nhãn kiểu `AnomalyNCD`).

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / cập nhật repo lên ổ local /content (chạy git trong 2 giây)
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b duytung/03-gc10-dac-trung-vung-loi https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin duytung/03-gc10-dac-trung-vung-loi
!git checkout duytung/03-gc10-dac-trung-vung-loi
!git reset --hard origin/duytung/03-gc10-dac-trung-vung-loi
!git status -sb

In [ ]:
# 3. Cài gói thiếu, đảm bảo JAX nhận GPU, và kiểm tra GC10-DET + thư mục nhãn XML
!pip uninstall -y -q torchao
!pip install -q continuum numpyro prettytable kagglehub

import os, glob, shutil, subprocess
has_gpu = subprocess.call(
    ['python', '-c', 'import jax; assert any(d.platform != "cpu" for d in jax.devices())']
) == 0
if not has_gpu:
    print('>>> JAX đang chạy CPU, thử cài jax[cuda12]...')
    !pip install -q -U "jax[cuda12]"

!python -c "import torch, jax, numpyro, continuum, prettytable, sklearn, scipy, datasets, peft; print('Imports OK! Torch CUDA:', torch.cuda.is_available(), '| JAX devices:', jax.devices())"

TARGET_GC10 = '/content/drive/MyDrive/Datasets/GC10-DET'
cands = [TARGET_GC10, '/content/drive/MyDrive/VB_CGCD_Work/GC10-DET', '/content/drive/MyDrive/VB_CGCD_Work/Datasets/GC10-DET']
found = [p for p in cands if os.path.isfile(f'{p}/1/img_06_425613700_00781.jpg') and len(glob.glob(f'{p}/label/*.xml')) > 0]

if not found:
    print('>>> Chưa thấy đủ GC10-DET + label/*.xml trên Drive, đang tự tìm/tải từ Kaggle...')
    import kagglehub
    os.makedirs(TARGET_GC10, exist_ok=True)
    for slug in ('lirick/gc10-det', 'alex000kim/gc10det'):
        try:
            dl_root = kagglehub.dataset_download(slug)
            print('Kiểm tra:', slug, '->', dl_root)
            if not os.path.isfile(f'{TARGET_GC10}/1/img_06_425613700_00781.jpg'):
                hits = glob.glob(f'{dl_root}/**/1/img_06_425613700_00781.jpg', recursive=True)
                if hits:
                    src_dir = os.path.dirname(os.path.dirname(hits[0]))
                    for sub in os.listdir(src_dir):
                        sp = os.path.join(src_dir, sub)
                        dp = os.path.join(TARGET_GC10, sub)
                        if os.path.isdir(sp) and sub.isdigit() and not os.path.exists(dp):
                            shutil.copytree(sp, dp)
            xmls = glob.glob(f'{dl_root}/**/*.xml', recursive=True)
            if xmls:
                os.makedirs(f'{TARGET_GC10}/label', exist_ok=True)
                for xp in xmls:
                    dst = os.path.join(TARGET_GC10, 'label', os.path.basename(xp))
                    if not os.path.exists(dst):
                        shutil.copy2(xp, dst)
                print(f'>>> Đã chép {len(xmls)} file XML từ {slug} vào {TARGET_GC10}/label/')
            if os.path.isfile(f'{TARGET_GC10}/1/img_06_425613700_00781.jpg') and len(glob.glob(f'{TARGET_GC10}/label/*.xml')) > 0:
                found = [TARGET_GC10]
                break
        except Exception as e:
            print('Thử', slug, 'thất bại:', e)

if found:
    n_xml = len(glob.glob(f'{found[0]}/label/*.xml'))
    print(f'>>> GC10-DET sẵn sàng tại: {found[0]} (có {n_xml} file XML trong label/)')
else:
    raise FileNotFoundError('Không tìm thấy file XML của GC10-DET trong các nguồn Kaggle.')

In [ ]:
# 4. Chạy ngầm Bước A1a (STAGE='auto' | 'eval')
STAGE = 'auto'

import os, subprocess, time

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a1a_bg.log'
PID_FILE = f'{WORK_DIR}/gc10_a1a_bg.pid'

running = False
if os.path.exists(PID_FILE):
    old_pid = open(PID_FILE).read().strip()
    if old_pid and os.path.exists(f'/proc/{old_pid}/status'):
        st = open(f'/proc/{old_pid}/status').read()
        if 'State:\tZ' not in st and 'State: Z' not in st:
            running = True
            print(f'>>> Pipeline đang chạy ngầm sẵn với PID {old_pid}. Chạy Cell 5 để xem log!')

if not running:
    proc = subprocess.Popen(
        ['nohup', 'bash', 'colab/run_gc10_a1a_bg.sh', STAGE],
        stdout=open(LOG_FILE, 'w'),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    with open(PID_FILE, 'w') as f:
        f.write(str(proc.pid))
    time.sleep(2)
    print(f'>>> Đã khởi chạy ngầm STAGE={STAGE}, PID={proc.pid}! Log: {LOG_FILE}')
    !head -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ, phần cứng, thời gian & bảng kết quả Bước A1a
import os

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a1a_bg.log'
PID_FILE = f'{WORK_DIR}/gc10_a1a_bg.pid'
SUMMARY_OUT = f'{WORK_DIR}/gc10_a1a_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = 'State:\tZ' not in st and 'State: Z' not in st
print(f'Trạng thái PID={pid}:', 'ĐANG CHẠY' if alive else 'ĐÃ DỪNG / XONG')

if os.path.exists(LOG_FILE):
    print('\n--- 40 dòng log gần nhất ---')
    !tail -n 40 {LOG_FILE}

if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ & THÔNG SỐ PHẦN CỨNG BƯỚC A1a ---')
    print(open(SUMMARY_OUT).read())